# Nuclear Waste Temperature — Spatial KNN
**CIVIL-226 — EPFL**

Same preprocessing as OG_improved, but using KNN (k=3) instead of Neural Network.
The 3 repetitions (power scenarios) are kept separate and predicted independently.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

## 2. Load data

In [ ]:
train   = pd.read_parquet('data/train.parquet')
test    = pd.read_parquet('data/test.parquet')
sensors = pd.read_parquet('data/sensors.parquet')

print(f'Train   : {train.shape}')
print(f'Test    : {test.shape}')
print(f'Sensors : {sensors.shape}')

## 3. Merge sensor positions

In [ ]:
train_full = train.merge(sensors, on='sensor', how='left')
print(train_full.head())
print(train_full.shape)

## 4. Clean missing values

In [ ]:
train_full = train_full.dropna()
print(train_full.shape)
print(train_full.columns.tolist())

## 5. Identify the 3 repetitions (power scenarios)

In [ ]:
# Each (sensor, time) pair has exactly 3 rows = 3 power scenarios
train_full['rep'] = train_full.groupby(['sensor', 'time']).cumcount()
print('Repetitions:', train_full['rep'].value_counts().sort_index().to_dict())
assert (train_full.groupby(['sensor','time']).size() == 3).all()
print('All (sensor, time) pairs have exactly 3 repetitions OK')

## 6. Cumulative energy feature

In [ ]:
# Cumulative energy per rep (each rep has its own power profile)
power_time = (
    train_full[['time', 'power', 'rep']]
    .drop_duplicates(subset=['time','rep'])
    .sort_values(['rep','time'])
    .copy()
)
power_time['dt']         = power_time.groupby('rep')['time'].diff().fillna(0)
power_time['cum_energy'] = power_time.groupby('rep').apply(
    lambda g: (g['power'] * g['dt']).cumsum()
).reset_index(level=0, drop=True)

train_full = train_full.merge(
    power_time[['time','rep','cum_energy']],
    on=['time','rep'], how='left'
)
print(f'NaN cum_energy: {train_full["cum_energy"].isna().sum()}')

## 7. Engineered features (same as OG_improved)

In [ ]:
eps = 1e-8
train_full['dist2']            = train_full['coor_x']**2 + train_full['coor_y']**2
train_full['power_over_dist2'] = train_full['power'] / (train_full['dist2'] + eps)
train_full['diffusion']        = train_full['dist2'] / (train_full['time'] + eps)

train_full = train_full.dropna(subset=['temperature'])
print(f'train_full shape: {train_full.shape}')

## 8. Train/validation split by sensor

In [ ]:
feature_cols = [
    'coor_x', 'coor_y', 'power', 'time',
    'cum_energy', 'power_over_dist2', 'diffusion'
]
target_col = 'temperature'

# Split by sensor (not by row) — same as OG_improved
unique_sensors = train_full['sensor'].unique()
rng = np.random.default_rng(42)
val_sensors   = rng.choice(unique_sensors, size=int(0.2 * len(unique_sensors)), replace=False)
train_sensors = np.setdiff1d(unique_sensors, val_sensors)

train_df = train_full[train_full['sensor'].isin(train_sensors)].copy()
val_df   = train_full[train_full['sensor'].isin(val_sensors)].copy()

assert set(train_df['sensor']).isdisjoint(set(val_df['sensor']))
print(f'Train sensors: {len(train_sensors)} | Val sensors: {len(val_sensors)}')
print(f'Train rows: {len(train_df)} | Val rows: {len(val_df)}')

## 9. Normalization

In [ ]:
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train = train_df[feature_cols].values
y_train = train_df[[target_col]].values
X_val   = val_df[feature_cols].values
y_val   = val_df[[target_col]].values

X_train_norm = scaler_X.fit_transform(X_train)
X_val_norm   = scaler_X.transform(X_val)
y_train_norm = scaler_y.fit_transform(y_train).ravel()
y_val_norm   = scaler_y.transform(y_val).ravel()

print(f'NaN X_train: {np.isnan(X_train_norm).sum()} | NaN y_train: {np.isnan(y_train_norm).sum()}')
print(f'X_train shape: {X_train_norm.shape}')

## 10. KNN (k=3)

Same features as OG_improved, normalized, k=3 neighbors with inverse distance weighting.
The 3 repetitions are included in training — the model learns the power-temperature relationship.

In [ ]:
K = 3
knn = KNeighborsRegressor(n_neighbors=K, weights='distance', n_jobs=-1)
knn.fit(X_train_norm, y_train_norm)
print(f'KNN (k={K}) trained on {len(X_train_norm)} samples')

## 11. Validation RMSE

In [ ]:
y_val_pred_norm = knn.predict(X_val_norm)
y_val_pred      = scaler_y.inverse_transform(y_val_pred_norm.reshape(-1,1)).ravel()
y_val_true      = scaler_y.inverse_transform(y_val_norm.reshape(-1,1)).ravel()

rmse_val = np.sqrt(mean_squared_error(y_val_true, y_val_pred))
print(f'KNN (k={K}) — RMSE validation : {rmse_val:.4f}')

# RMSE per repetition
for rep in [0, 1, 2]:
    mask = val_df['rep'].values == rep
    rmse_rep = np.sqrt(mean_squared_error(y_val_true[mask], y_val_pred[mask]))
    print(f'  Rep {rep} RMSE : {rmse_rep:.4f}')

## 12. Final predictions & Submission

In [ ]:
# Reload and preprocess test set — same pipeline as train
test_orig   = pd.read_parquet('data/test.parquet')
sensors_df  = pd.read_parquet('data/sensors.parquet')
sensors_df  = sensors_df.drop_duplicates(subset='sensor', keep='first')

test_full = test_orig.merge(sensors_df, on='sensor', how='left')

# Repetition number
test_full['rep'] = test_full.groupby(['sensor','time']).cumcount()

# Cumulative energy per rep
power_time_test = (
    test_full[['time','power','rep']]
    .drop_duplicates(subset=['time','rep'])
    .sort_values(['rep','time'])
    .copy()
)
power_time_test['dt']         = power_time_test.groupby('rep')['time'].diff().fillna(0)
power_time_test['cum_energy'] = power_time_test.groupby('rep').apply(
    lambda g: (g['power'] * g['dt']).cumsum()
).reset_index(level=0, drop=True)

test_full = test_full.merge(
    power_time_test[['time','rep','cum_energy']],
    on=['time','rep'], how='left'
)

# Engineered features
eps = 1e-8
test_full['dist2']            = test_full['coor_x']**2 + test_full['coor_y']**2
test_full['power_over_dist2'] = test_full['power'] / (test_full['dist2'] + eps)
test_full['diffusion']        = test_full['dist2'] / (test_full['time'] + eps)

print(f'test_orig : {len(test_orig)} | test_full : {len(test_full)}')
assert len(test_full) == len(test_orig), 'Row count mismatch!'

# Predict
X_test      = test_full[feature_cols].values
X_test_norm = scaler_X.transform(X_test)
y_test_norm = knn.predict(X_test_norm)
y_test_pred = scaler_y.inverse_transform(y_test_norm.reshape(-1,1)).ravel()

# Submission
submission = pd.DataFrame({
    'Id'         : np.arange(len(test_orig), dtype=int),
    'temperature': y_test_pred.astype(float)
})

assert list(submission.columns) == ['Id', 'temperature']
assert len(submission) == len(test_orig)
assert np.isfinite(submission['temperature']).all()
assert submission.isna().sum().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved — {len(submission)} rows')
print(f'Min: {y_test_pred.min():.2f} | Max: {y_test_pred.max():.2f} | Mean: {y_test_pred.mean():.2f}')
display(submission.head())

## 13. Visualisation — predictions vs train

In [ ]:
test_full['temperature_pred'] = y_test_pred
test_full['time_years']       = test_full['time'] / (365.25 * 24 * 3600)
train_full['time_years']      = train_full['time'] / (365.25 * 24 * 3600)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for rep in [0, 1, 2]:
    # Temperature
    tr = train_full[train_full['rep']==rep].groupby('time_years')['temperature'].mean()
    te = test_full[test_full['rep']==rep].groupby('time_years')['temperature_pred'].mean()
    axes[0, rep].plot(tr.index, tr.values, color='tomato',    linewidth=0.8, label='Train (true)')
    axes[0, rep].plot(te.index, te.values, color='steelblue', linewidth=0.8, label='Test (pred)')
    axes[0, rep].set_title(f'Temperature — Rep {rep}')
    axes[0, rep].set_xlabel('Time (years)'); axes[0, rep].set_ylabel('Temperature (°C)')
    axes[0, rep].legend()

    # Power
    tr_p = train_full[train_full['rep']==rep].groupby('time_years')['power'].mean()
    te_p = test_full[test_full['rep']==rep].groupby('time_years')['power'].mean()
    axes[1, rep].plot(tr_p.index, tr_p.values, color='orange', linewidth=0.8, label='Train power')
    axes[1, rep].plot(te_p.index, te_p.values, color='green',  linewidth=0.8, label='Test power')
    axes[1, rep].set_title(f'Power — Rep {rep}')
    axes[1, rep].set_xlabel('Time (years)'); axes[1, rep].set_ylabel('Power (W)')
    axes[1, rep].legend()

plt.tight_layout()
plt.show()